# 🏆 사장님, 전설이 되다 — 창업 법률·세무 RAG 챗봇

> **고려대학교 세종캠퍼스 · 데이터사이언스세미나 · 2026년 1학기**



처음 창업하는 사장님이 **사업자등록·세금·인허가·세액감면·정부지원사업, 그리고 23개 업종별
창업 절차**에 대해 무엇이든 물어보면, 국세청·중소벤처기업부·법제처 **공식 자료(문서 726건)를
검색해 근거와 출처를 들어 답하고, 자료에 없는 내용은 지어내지 않고 모른다고 말하는**
RAG(Retrieval-Augmented Generation) 챗봇을 구현합니다.

| 구성 요소 | 사용 기술 |
|---|---|
| 형태소 분석(토큰화) | `kiwipiepy` |
| 임베딩 | `jhgan/ko-sroberta-multitask` (한국어 특화 Sentence-BERT) |
| 벡터 검색(리트리버) | `FAISS` (코사인 유사도) |
| 답변 생성(LLM) | OpenAI `gpt-4o-mini` *(API 키 없으면 검색 결과 안내 모드로 동작)* |
| UI | `Gradio` |

**목차**
1. 문제 정의 및 사례 조사
2. 데이터 수집 및 기초 텍스트 분석
3. RAG 챗봇 구성 (청킹 → 임베딩 → 리트리버 → 생성 → 하이퍼파라미터 튜닝)
4. Gradio UI 챗봇 구현
5. 테스트 및 챗봇 성능 개선 (할루시네이션 방지)
6. 결론 및 한계

```bash
# 필요 패키지 설치 (최초 1회)
pip install pandas matplotlib seaborn kiwipiepy sentence-transformers faiss-cpu openai gradio
```

---
## 1. 문제 정의 및 사례 조사

### 1.1 어떤 문제를 풀고자 하는가?

처음 사업을 시작하는 사장님은 누구나 이런 질문 앞에서 막막해집니다.

> *"블로그에서 물건 좀 팔아보려는데 사업자등록을 해야 하나?"*, *"간이과세자가 뭐지?"*,
> *"부가세 신고는 언제 하지?"*, *"청년 창업이면 세금 깎아준다던데 나도 해당되나?"*

- **정보가 기관별로 흩어져 있다** — 사업자등록·세금은 국세청(홈택스), 통신판매업 신고는 정부24,
  세액감면은 조세특례제한법, 지원사업은 K-Startup… 한 가지 일을 하려면 여러 기관 자료를 뒤져야 합니다.
- **용어가 어렵다** — 간이과세자, 공급대가, 기준경비율, 수도권과밀억제권역… 처음 듣는 행정·세무 용어가
  검색 자체를 가로막습니다.
- **모르면 실제로 돈을 잃는다** — 사업자 미등록 시 공급가액의 1% 가산세, 현금영수증 미발급 시 20% 가산세,
  청년창업 세액감면(최대 100%, 5년)을 몰라서 못 받으면 수백만 원 손해. **정확성이 곧 돈인 도메인**입니다.
- **전문가 상담은 비싸고 느리다** — 세무사 상담은 비용 부담이 있고, 국세상담센터(126)는 대기가 깁니다.

### 1.2 왜 일반 챗봇(ChatGPT)으로는 안 되는가?

범용 LLM은 **한국 세법·지원사업의 세부 기준을 정확히 모르면서도 그럴듯하게 답하는
할루시네이션(hallucination)** 이 발생합니다. 게다가 세법 기준금액과 지원사업 요건은 **매년 바뀌는데**
(예: 간이과세 기준 1억 4백만원은 2024년 이후 적용분) LLM의 학습 시점은 과거에 멈춰 있습니다.
가산세·과태료로 직결되는 영역에서 출처 없는 답변은 쓸 수 없습니다.

### 1.3 사례 조사

- **공공 상담 서비스**: 국세청 홈택스 상담·국세상담센터(126), 정부24, 중소기업 통합콜센터(1357) 등이
  운영되지만 전화 대기·게시판 답변 지연이 있고, 챗봇류는 대부분 **미리 정의된 시나리오(룰베이스)** 방식이라
  자유로운 질문에는 약합니다.
- **민간 세무 플랫폼**: 세무 신고 대행 앱들이 성장할 만큼 초보 사업자의 세무 정보 수요는 검증되어 있습니다.
  다만 신고 대행이 중심이고, 창업 전 단계(등록·인허가·감면·지원사업)를 아우르는 안내는 부족합니다.



### 1.4 해결 방법: 왜 RAG인가?

**RAG(Retrieval-Augmented Generation)** 는 질문과 관련된 공식 문서를 **검색(Retrieval)** 해서 LLM에게
근거 자료로 제공하고, 그 안에서만 답하게 **생성(Generation)** 하는 방식입니다.

| 비교 항목 | 룰베이스 챗봇 | LLM 단독 | **RAG 챗봇 (우리 방식)** |
|---|---|---|---|
| 자유로운 질문 이해 | ❌ | ✅ | ✅ |
| 공식 자료 기반의 정확한 답 | ✅ (등록된 것만) | ❌ 할루시네이션 | ✅ 검색 근거 기반 |
| 법령 개정 시 업데이트 | 시나리오 재작성 | 재학습 필요 | **데이터 파일만 교체** |
| 답변 근거(출처) 제시 | △ | ❌ | ✅ |

### 1.5 챗봇의 목표와 성공 기준

> **"국세청·중기부 공식 자료 안에 있는 질문에는 정확하게 답하고, 없는 질문에는 모른다고 말하며,
> 항상 출처와 '최신 기준 확인' 안내를 함께 제시하는 창업 안내 챗봇"**

| 성공 기준 | 검증 방법 (→ 5장 테스트) |
|---|---|
| ① 데이터에 있는 질문에 정확히 답변 | 평가셋 18문항 검색 정확도(Hit@k, MRR) + 정답 키워드 포함률 |
| ② 데이터에 없는 질문은 정중히 거절 + 전문기관 안내 | 도메인 외 질문 테스트 (할루시네이션 방지) |
| ③ 답변 근거(출처)와 시점 한계 고지 | 모든 답변에 출처 + 최신 확인 안내 자동 표시 |

---
## 2. 데이터 수집 및 기초 텍스트 분석

### 2.1 데이터셋 소개 및 구축 과정 (v1 → v2 보강 이력 포함)

- **원자료 (모두 공공기관 공식 자료)**:
  1. 국세청 「SNS마켓 사업자 신종업종 세무안내」 — 사업자등록·과세유형·부가세·종소세·현금영수증·통신판매업 신고
  2. 국세청 「창업중소기업 세액감면」 안내 + 법제처 찾기쉬운 생활법령정보 — 조세특례제한법 제6조
  3. 중소벤처기업부 「2026년 창업지원사업 통합공고」(제2025-648호) — 예비창업패키지 등 지원사업
  4. **법제처 찾기쉬운 생활법령정보 「업종별 창업 가이드」 23종** (2026.5 기준) — 음식점·카페·네일샵·펜션·푸드트럭·인터넷쇼핑몰·동업계약·소상공인 지원 등
- **데이터 보강 이력 (테스트 → 수집의 선순환)**: v1(핵심 세무·감면 30건)으로 만든 챗봇을 테스트한 결과
  **직원 고용(4대보험)·가게 입지·임대차 질문이 거절되는 공백을 발견** → 법제처 업종가이드를 크롤링해
  v2로 보강했습니다. *"거절된 질문 로그가 다음 수집 목록이 된다"* 는 설계를 실제로 한 바퀴 돌린 결과입니다.
- **구축 절차**:
  1. 조원이 원자료를 크롤링·정리해 청크 단위 JSONL로 구축 (`data/chunks_v2.jsonl`, 청크 3,089개)
  2. `노트북의 데이터 복원 셀`로 **같은 소주제의 청크를 문서 단위로 복원** — 청크 간 겹침(overlap)은
     suffix-prefix 매칭으로 중복 제거, 크롤링 공백 노이즈는 정규화 → **문서 726건 / 31개 카테고리**
  3. 문서마다 **원자료 출처(`source`)와 기준 시점을 메타데이터로 보존** → 챗봇이 답변에 출처를 표시하는 데 사용
- **컬럼**: `doc_id` / `category` / `router_tag` / `field`(업종) / `title` / `content` / `source`
- **기준 시점**: 법제처 자료 2026.5, 통합공고 2026년, 세법 2024년 이후 기준 — 법령·요건은 개정되므로
  챗봇이 모든 답변에 "최신 기준 확인" 안내를 자동으로 덧붙이도록 설계 (3.4절)

> 💡 조원이 `chunks_vN.jsonl`을 업데이트하면 `python 노트북의 데이터 복원 셀` 한 번으로
> 데이터셋이 갱신되고, 노트북 전체 파이프라인이 그대로 동작합니다.

In [ ]:
import os, platform, warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 60)

# ---- 한글 폰트 설정 (운영체제별) ----
font_family = {"Darwin": "AppleGothic", "Windows": "Malgun Gothic"}.get(platform.system(), "NanumGothic")
sns.set_theme(style="whitegrid", font=font_family)
plt.rcParams["axes.unicode_minus"] = False

# ---- 실험 설정 (하이퍼파라미터) ----
DATA_PATH       = "data/dataset.csv"
EMBEDDING_MODEL = "jhgan/ko-sroberta-multitask"   # 한국어 특화 문장 임베딩 모델
LLM_MODEL       = "gpt-4o-mini"                   # 답변 생성용 LLM
CHUNK_SIZE      = 180     # 청크 최대 길이(글자 수) → 3.5에서 튜닝
TOP_K           = 3       # 검색할 문서(청크) 수      → 3.5에서 튜닝
SIM_THRESHOLD   = 0.45    # 유사도 임계값(미만이면 답변 거절) — 초기값, 3.5(c) 분포 분석으로 확정

# ---- OpenAI API 키는 환경변수로만 입력 ----
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")

print(f"OpenAI API 키 설정 여부: {'✅ 설정됨 (LLM 생성 모드)' if OPENAI_API_KEY else '⚠️ 없음 (검색 결과 안내 모드)'}")

#### 📦 데이터 구축 파이프라인 — 원본 PDF → 청크 → 데이터셋

법제처·국세청·중기부의 공식 자료(업종별 창업 가이드 PDF 등)를 다음 순서로 가공했습니다.

1. **텍스트 추출** — `pdftotext -layout`로 PDF 본문 추출
2. **정제** — 페이지 번호·목차 점선·저작권 안내 등 **보일러플레이트 제거**
3. **섹션 분할** — 번호 헤딩(`1.2.1 …`) 기준으로 *(업종 - 섹션)* 단위 분리
4. **청킹** — 문장 경계를 지키며 **600자 청크**로 묶고, 청크 경계에서 **80자를 겹쳐(overlap)** 문맥 보존. 문장부호 없는 초장문은 글자 수로 강제 분할
5. **병합** — 기존 세무·감면 청크 + 신규 업종 가이드 청크 → `chunks_v2.jsonl`

> ⚙️ 아래 첫 셀은 위 파이프라인의 **실제 코드**입니다. 원본 PDF와 `pdftotext`(poppler-utils)가 있으면
> `RUN_CHUNKING=True`로 직접 재현할 수 있고, 원본 PDF가 없는 환경에서는 **미리 구축해 동봉한
> `data/chunks_v2.jsonl`을 그대로 사용**합니다(노트북 전체가 끊김 없이 실행됩니다).

In [ ]:
# ── 데이터 구축 ①: PDF 청킹 파이프라인 ─────────────────────────────
import re, json, glob, subprocess, math, statistics

PDF_DIR     = "data/pdfs"               # 업종 가이드 PDF 폴더 (제출본에는 비어 있을 수 있음)
EXIST_JSONL = "data/chunks_team.jsonl"  # 기존 세무·감면 청크
OUT_JSONL   = "data/chunks_v2.jsonl"    # 병합 결과(정본)
MAX_CHARS, OVERLAP_CHARS, MIN_CHARS = 600, 80, 25
RUN_CHUNKING = False   # 원본 PDF + pdftotext가 있으면 True로 바꿔 직접 청킹을 재현

def est_tokens(t): return math.ceil(len(t) / 1.8)
def extract_text(pdf):
    return subprocess.run(["pdftotext", "-layout", pdf, "-"], capture_output=True, text=True).stdout
def field_label(path):
    stem = re.sub(r"^pdf", "", os.path.basename(path)[:-4])
    return stem.replace("_", " ").replace("ㆍ", "·").strip()

DOTLEADER = re.compile(r"\.(\s*\.){3,}")          # 목차 점선
PAGEMARK  = re.compile(r"^\s*\d+\s*/\s*\d+\s*$")  # 'X / 32'
HEADING   = re.compile(r"^\s*(\d+(?:\.\d+)*)\.?\s+(\S.*)$")
BOILER_KEYS = ("찾기쉬운 생활법령", "찾기 쉬운 생활법령정보", "이 정보는", "공공데이터 정책",
               "저작권", "법제처 법제정보", "유권해석", "국민 신문고")

def clean_lines(raw):
    out = []
    for ln in raw.split("\n"):
        s = ln.replace("\f", "").strip()
        if not s or PAGEMARK.match(s) or DOTLEADER.search(s): continue
        if any(k in s for k in BOILER_KEYS): continue
        out.append(s)
    return out

def parse_sections(lines, field):
    sections, cur_title, buf = [], None, []
    def flush():
        if cur_title and buf:
            txt = " ".join(buf).strip()
            if len(txt) >= 25: sections.append((cur_title, txt))
    for s in lines:
        m = HEADING.match(s)
        if m and len(m.group(2)) <= 40 and not m.group(2).endswith(("다.", "요.", "다", "음")):
            flush(); buf = []
            cur_title = f"{field} - {m.group(2).strip()}"
        elif cur_title is not None:
            buf.append(s)
    flush()
    return sections

def split_sentences(text):
    parts = re.split(r'(?<=[다요음임함됨)])\.\s+|(?<=\.)\s+', text)
    return [p.strip() for p in parts if p.strip()]

def hard_wrap(s):                                  # 문장 경계 없는 초장문 → 글자수 강제 분할(80자 겹침)
    if len(s) <= MAX_CHARS: return [s]
    out, i, step = [], 0, MAX_CHARS - OVERLAP_CHARS
    while i < len(s):
        out.append(s[i:i + MAX_CHARS]); i += step
    return out

def recursive_split(text):                         # 문장 단위로 600자 청크 + 80자 overlap
    if len(text) <= MAX_CHARS: return [text]
    sents, chunks, cur = split_sentences(text), [], []
    for s in sents:
        for seg in hard_wrap(s):
            cur.append(seg)
            if len(" ".join(cur)) >= MAX_CHARS:
                chunks.append(" ".join(cur))
                keep, acc = [], 0
                for ss in reversed(cur):
                    acc += len(ss); keep.insert(0, ss)
                    if acc >= OVERLAP_CHARS: break
                cur = keep[:]
    if cur: chunks.append(" ".join(cur))
    final = []
    for c in chunks:
        final.extend(hard_wrap(c) if len(c) > MAX_CHARS else [c])
    return [c for c in final if c.strip()]

def make_chunks():
    records = []
    for pdf in sorted(glob.glob(os.path.join(PDF_DIR, "*.pdf"))):
        field, src = field_label(pdf), os.path.basename(pdf)
        for title, body in parse_sections(clean_lines(extract_text(pdf)), field):
            for piece in recursive_split(body):
                piece = piece.strip()
                if len(piece) < MIN_CHARS: continue
                records.append({"field": field, "section": title, "source": src, "text": piece,
                                "n_chars": len(piece), "n_tokens_est": est_tokens(piece)})
    return records

def build_chunks_v2():
    existing = [json.loads(l) for l in open(EXIST_JSONL, encoding="utf-8") if l.strip()] if os.path.exists(EXIST_JSONL) else []
    merged = existing + make_chunks()
    for i, c in enumerate(merged): c["id"] = i
    with open(OUT_JSONL, "w", encoding="utf-8") as f:
        for c in merged: f.write(json.dumps(c, ensure_ascii=False) + "\n")
    n = [c["n_chars"] for c in merged]
    print(f"청킹 완료: 총 {len(merged)} 청크 → {OUT_JSONL} (평균 {statistics.mean(n):.0f}자)")
    return merged

_have_pdf = bool(glob.glob(os.path.join(PDF_DIR, "*.pdf")))
_have_tool = subprocess.run(["which", "pdftotext"], capture_output=True).returncode == 0
if RUN_CHUNKING and _have_pdf and _have_tool:
    build_chunks_v2()
else:
    print(f"원본 PDF/pdftotext 미검출 → 동봉된 {OUT_JSONL} 사용 (위 함수는 데이터 구축 방법을 보여주는 재현 코드)")

In [ ]:
# ── 데이터 구축 ②: 청크(jsonl) → 문서 단위 데이터셋(csv) 복원 ──────────
# 같은 (분류, 섹션)에 속한 청크들을 한 문서로 병합하고, 청킹 overlap(겹침)은
# suffix-prefix 매칭으로 중복 제거 → RAG 입력용 문서 단위 dataset.csv 생성.
from collections import OrderedDict

def _norm(t): return re.sub(r"\s+", " ", str(t)).strip()
def _merge_overlap(a, b, max_ov=400):
    k = min(len(a), len(b), max_ov)
    while k > 0:
        if a[-k:] == b[:k]: return a + b[k:]
        k -= 1
    return a + " " + b

_rows = [json.loads(l) for l in open(OUT_JSONL, encoding="utf-8") if l.strip()]
_docs = OrderedDict()
for r in _rows:
    cat = r.get("category") or (f"업종가이드 - {r['field']}" if r.get("field") else "기타")
    key = (cat, r.get("section", ""))
    d = _docs.setdefault(key, {"category": cat, "section": r.get("section", ""),
                               "router_tag": r.get("router_tag", "업종가이드"),
                               "field": r.get("field", ""), "source": r.get("source", "미상"), "texts": []})
    d["texts"].append(_norm(r["text"]))

_recs = []
for d in _docs.values():
    content = d["texts"][0]
    for t in d["texts"][1:]:
        content = _merge_overlap(content, t)
    _recs.append({"doc_id": len(_recs) + 1, "category": re.sub(r"^\d+\.\s*", "", d["category"]),
                  "router_tag": d["router_tag"], "field": d["field"],
                  "title": d["section"], "content": content, "source": d["source"]})

pd.DataFrame(_recs).to_csv(DATA_PATH, index=False, encoding="utf-8")
print(f"청크 {len(_rows):,}개 → 문서 {len(_recs)}건 복원 → {DATA_PATH}")

In [ ]:
# 데이터 로드
df = pd.read_csv(DATA_PATH)
print(f"문서 수: {len(df)}건")
df.head()

### 2.2 기초 통계 분석

본격적인 토큰화에 앞서, 데이터가 어떤 분포를 갖는지 확인합니다.
**문서 길이 분포는 이후 청킹(chunking) 크기를 정하는 근거**가 됩니다.

In [ ]:
# 카테고리 분포와 문서 길이 분포
df["n_chars"] = df["content"].str.len()

fig, axes = plt.subplots(1, 2, figsize=(13, max(4, 0.28 * df["category"].nunique())))

cat_counts = df["category"].value_counts()
sns.barplot(x=cat_counts.values, y=cat_counts.index, ax=axes[0], color="#4C72B0")
axes[0].set_title("카테고리별 문서 수")
axes[0].set_xlabel("문서 수")

sns.histplot(df["n_chars"], bins=12, ax=axes[1], color="#55A868")
axes[1].axvline(df["n_chars"].mean(), color="red", ls="--", label=f"평균 {df['n_chars'].mean():.0f}자")
axes[1].set_title("문서 길이(글자 수) 분포")
axes[1].set_xlabel("글자 수")
axes[1].legend()

plt.tight_layout()
plt.show()

print(df["n_chars"].describe().round(1))

### 2.3 토큰화 (형태소 분석)

한국어는 영어와 달리 **조사·어미가 단어에 붙는 교착어**라서, 단순히 띄어쓰기(어절)로 자르면
`신고는`, `신고를`, `신고하면` 이 전부 다른 토큰이 되어 빈도 분석이 왜곡됩니다.
따라서 **형태소 분석기(`kiwipiepy`)** 로 토큰화하여 의미 단위(명사 등)를 추출합니다.

> `kiwipiepy`는 순수 파이썬 패키지로 Java 설치가 필요한 KoNLPy보다 설치가 간편하고 속도도 빠릅니다.

In [ ]:
from kiwipiepy import Kiwi

kiwi = Kiwi()

# 어절 토큰화 vs 형태소 토큰화 비교
sample = "사업을 시작한 날로부터 20일 이내에 국세청 홈택스에서 온라인으로 신청하거나, 세무서 민원봉사실을 방문해 신청합니다."
eojeol_tokens = sample.split()
morph_tokens = [f"{t.form}/{t.tag}" for t in kiwi.tokenize(sample)]

print("【어절 토큰화】 (단순 띄어쓰기)")
print(eojeol_tokens)
print(f"\n【형태소 토큰화】 (kiwipiepy)")
print(morph_tokens)
print("\n→ 어절 토큰화는 '신청하거나'와 '신청합니다'를 다른 토큰으로 세지만,")
print("   형태소 토큰화는 둘 다 '신청/NNG'으로 묶여 의미 단위 분석이 가능합니다.")

In [ ]:
# 전체 데이터에서 명사(NNG: 일반명사, NNP: 고유명사)만 추출하여 빈도 분석
stopwords = {"경우", "이내", "이상", "미만", "이전", "이후", "해당", "기준", "방법", "대상", "일부"}

def extract_nouns(text):
    return [t.form for t in kiwi.tokenize(text)
            if t.tag in ("NNG", "NNP") and len(t.form) >= 2 and t.form not in stopwords]

all_nouns = [n for content in df["content"] for n in extract_nouns(content)]
noun_freq = Counter(all_nouns)

print(f"전체 명사 토큰 수: {len(all_nouns):,}개 / 고유 명사 수: {len(noun_freq):,}개")

top20 = pd.DataFrame(noun_freq.most_common(20), columns=["명사", "빈도"])
plt.figure(figsize=(9, 6))
sns.barplot(data=top20, x="빈도", y="명사", color="#4C72B0")
plt.title("창업 법률·세무 문서 최다 빈출 명사 TOP 20")
plt.tight_layout()
plt.show()

**분석 결과 해석**: `사업자`, `창업`, `감면`, `신고`, `과세` 등 창업 법률·세무의 핵심 용어가
상위에 분포 → 데이터가 목표 도메인을 잘 대표하고 있음을 확인할 수 있습니다.
이 빈출 키워드들은 5장에서 **테스트 질문을 설계하는 근거**로도 활용됩니다.

### 2.4 임베딩 모델 관점의 토큰 길이 분석 → 청킹 설계 근거

임베딩 모델은 입력 길이 제한이 있습니다(`ko-sroberta`는 **최대 128 서브워드 토큰**).
문서가 이 제한을 넘으면 **뒷부분이 잘린 채 임베딩**되어 검색 품질이 떨어집니다.
우리 문서들이 제한을 넘는지 확인해 봅니다.

In [ ]:
from transformers import AutoTokenizer
from transformers import logging as hf_logging
hf_logging.set_verbosity_error()   # 긴 문서 인코딩 시 길이 경고 숨김 (정보성 경고일 뿐 동작과 무관)

hf_tokenizer = AutoTokenizer.from_pretrained(EMBEDDING_MODEL)
MAX_SEQ_LEN = 128  # ko-sroberta-multitask의 최대 입력 길이

df["n_tokens"] = df["content"].apply(lambda t: len(hf_tokenizer.encode(t)))

plt.figure(figsize=(9, 4))
sns.histplot(df["n_tokens"], bins=15, color="#C44E52")
plt.axvline(MAX_SEQ_LEN, color="black", ls="--", lw=2, label=f"모델 입력 한계 ({MAX_SEQ_LEN} 토큰)")
plt.title("문서별 서브워드 토큰 수 분포 vs 임베딩 모델 입력 한계")
plt.xlabel("서브워드 토큰 수")
plt.legend()
plt.tight_layout()
plt.show()

n_over = (df["n_tokens"] > MAX_SEQ_LEN).sum()
print(f"입력 한계(128토큰) 초과 문서: {n_over}건 / {len(df)}건 ({n_over/len(df)*100:.0f}%)")
if n_over > 0:
    print("→ 긴 문서를 통째로 임베딩하면 뒷부분이 잘려 정보가 손실됩니다. '청킹(분할)'이 필요합니다! (3.1절)")
else:
    print("→ 현재 데이터는 모두 한계 이내지만, 법령 전문 등으로 확장하면 초과 문서가 생기므로 청킹 파이프라인을 갖춥니다. (3.1절)")

---
## 3. RAG 챗봇 구성

```
[사장님의 질문]
     │
     ▼
① 질문 임베딩 (ko-sroberta) ──┐
                               ▼
② FAISS 벡터 검색 ◀── [창업 법률·세무 청크 임베딩 DB]  ← 사전 구축
     │  top-k 청크 + 유사도 점수
     ▼
③ 유사도 임계값 검사 ─── 미달 ──▶ "자료에 없습니다" + 전문기관 안내 (할루시네이션 방지)
     │ 통과
     ▼
④ 프롬프트 구성 (참고자료 + 질문 + 가드레일 규칙)
     ▼
⑤ LLM(gpt-4o-mini) 답변 생성 ──▶ [답변 + 출처 + 최신 기준 확인 안내]
```

### 3.1 청킹 (Chunking)

2.4절에서 확인했듯 일부 문서는 임베딩 모델의 입력 한계(128토큰)를 초과합니다.
문서를 **문장 단위로 자른 뒤 일정 길이(`CHUNK_SIZE`자) 이하로 묶고**,
문맥이 끊기지 않도록 **이웃 청크와 1문장씩 겹치게(overlap)** 만듭니다.
또한 각 청크 앞에 **문서 제목을 붙여** 임베딩 시 주제 정보가 보존되도록 합니다.

> ⚙️ **크롤링 데이터 대응**: 법제처 자료에는 표를 텍스트로 펴면서 생긴 **문장부호 없는 초장문**
> (수천 자가 한 '문장')이 존재합니다. 문장 단위 분할만 쓰면 이런 텍스트가 통째로 한 청크가 되어
> 임베딩이 잘리므로, **문장이 청크 한도를 넘으면 강제 분할(30자 겹침)** 하는 단계를 추가했습니다.

In [ ]:
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap_sents=1):
    """문장 경계를 지키면서 chunk_size(글자) 이하로 분할. 이웃 청크와 overlap_sents 문장 중첩.
    문장부호가 없는 초장문(표를 펴낸 크롤링 텍스트 등)은 chunk_size 단위로 강제 분할."""
    # ① 문장 분리 + 한도 초과 문장은 강제 분할 (30자 겹침으로 문맥 연결)
    sents = []
    for s in kiwi.split_into_sents(text):
        t = s.text.strip()
        while len(t) > chunk_size:
            sents.append(t[:chunk_size])
            t = t[chunk_size - 30:]
        if t:
            sents.append(t)
    # ② 청크 패킹: chunk_size 예산 안에서 문장 누적 (겹침 문장도 예산을 넘으면 생략)
    chunks, cur, cur_len = [], [], 0
    for s in sents:
        if cur and cur_len + len(s) > chunk_size:
            chunks.append(" ".join(cur))
            keep = cur[-overlap_sents:] if overlap_sents else []
            if sum(len(x) for x in keep) + len(s) > chunk_size:
                keep = []                      # 겹침까지 넣으면 한도 초과 → 겹침 생략
            cur, cur_len = keep, sum(len(x) for x in keep)
        cur.append(s)
        cur_len += len(s)
    if cur:
        chunks.append(" ".join(cur))
    return chunks

def make_chunks(df, chunk_size=CHUNK_SIZE):
    """데이터프레임 전체를 청크 단위 데이터프레임으로 변환"""
    rows = []
    for _, r in df.iterrows():
        for j, ch in enumerate(chunk_text(r["content"], chunk_size)):
            rows.append({"doc_id": r["doc_id"], "category": r["category"], "title": r["title"],
                         "source": r["source"], "chunk_no": j, "chunk": ch})
    return pd.DataFrame(rows)

chunk_df = make_chunks(df, CHUNK_SIZE)
print(f"원본 문서 {len(df)}건 → 청크 {len(chunk_df)}개 (chunk_size={CHUNK_SIZE}자)")
chunk_df.head(6)

### 3.2 임베딩 (Embedding)

**임베딩**은 문장을 의미를 담은 고차원 벡터로 변환하는 과정입니다.
의미가 비슷한 문장일수록 벡터 공간에서 가까워(코사인 유사도가 높아)집니다.
한국어 문장 유사도 태스크로 학습된 **`jhgan/ko-sroberta-multitask`** (768차원)를 사용합니다.

먼저 임베딩이 '의미'를 잡아내는지 간단히 검증해 봅니다.

In [ ]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer(EMBEDDING_MODEL)
print(f"모델: {EMBEDDING_MODEL} | 벡터 차원: {embedder.get_sentence_embedding_dimension()} | 최대 입력: {embedder.max_seq_length} 토큰")

# 임베딩이 의미적 유사성을 포착하는지 확인
demo_sents = [
    "사업자등록은 언제까지 해야 하나요?",
    "사업자 신고 기한이 궁금해요",      # 표현은 다르지만 의미가 같은 문장
    "부가가치세 신고는 언제 하나요?",   # 같은 도메인, 다른 주제
    "오늘 점심 뭐 먹을까?",             # 완전히 다른 주제
]
demo_emb = embedder.encode(demo_sents, normalize_embeddings=True)
sim_matrix = demo_emb @ demo_emb.T  # 정규화된 벡터의 내적 = 코사인 유사도

plt.figure(figsize=(7, 5.5))
sns.heatmap(sim_matrix, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=[s[:13] for s in demo_sents], yticklabels=[s[:13] for s in demo_sents])
plt.title("문장 간 코사인 유사도: 임베딩은 '단어'가 아닌 '의미'를 본다")
plt.tight_layout()
plt.show()
print("→ '사업자등록 기한'과 '사업자 신고 기한'은 표현이 달라도 유사도가 가장 높게 측정됩니다.")

### 3.3 벡터 인덱스 & 리트리버 (Retriever)

모든 청크를 임베딩해 **FAISS** 인덱스에 저장하고, 질문이 들어오면
질문 임베딩과 **코사인 유사도가 높은 상위 k개 청크**를 찾아주는 `Retriever`를 구현합니다.
(벡터를 정규화한 뒤 내적(`IndexFlatIP`)을 쓰면 코사인 유사도와 동일합니다.)

In [ ]:
import faiss

class Retriever:
    """FAISS 기반 의미 검색 리트리버"""
    def __init__(self, chunk_df, embedder):
        self.chunk_df = chunk_df.reset_index(drop=True)
        self.embedder = embedder
        # 제목을 청크 앞에 붙여 주제 정보를 보존한 채 임베딩
        texts = (self.chunk_df["title"] + ": " + self.chunk_df["chunk"]).tolist()
        emb = embedder.encode(texts, normalize_embeddings=True, show_progress_bar=False)
        self.index = faiss.IndexFlatIP(emb.shape[1])           # 내적(=코사인 유사도) 인덱스
        self.index.add(np.asarray(emb, dtype="float32"))

    def search(self, query, top_k=3):
        q_emb = self.embedder.encode([query], normalize_embeddings=True)
        scores, idxs = self.index.search(np.asarray(q_emb, dtype="float32"), top_k)
        results = []
        for s, i in zip(scores[0], idxs[0]):
            if i == -1:
                continue
            row = self.chunk_df.iloc[int(i)]
            results.append({"doc_id": int(row["doc_id"]), "category": row["category"],
                            "title": row["title"], "source": row["source"],
                            "chunk": row["chunk"], "score": float(s)})
        return results

retriever = Retriever(chunk_df, embedder)
print(f"인덱스 구축 완료: 청크 {retriever.index.ntotal}개\n")

# 검색 데모
for q in ["사업자등록 안 하면 어떻게 돼?", "나라에서 창업할 때 지원해주는 거 있어?"]:
    print(f"Q: {q}")
    for h in retriever.search(q, top_k=3):
        print(f"   [{h['score']:.3f}] ({h['category']}) {h['title']} — {h['chunk'][:40]}...")
    print()

**확인 포인트**: "나라에서 지원해주는 거"처럼 데이터에 없는 표현으로 물어도
의미가 가장 가까운 **창업 지원사업** 문서를 찾아냅니다 — 키워드 검색이 아닌 **의미 검색**의 힘입니다.

### 3.4 프롬프트 설계 & 답변 생성 (Generation)

검색된 청크를 **[참고 자료]** 로 프롬프트에 넣고 LLM이 그 안에서만 답하게 합니다.
시스템 프롬프트에 **할루시네이션 방지 가드레일**을 명시하는 것이 핵심입니다:

1. 참고 자료에 있는 내용**만** 근거로 답변
2. 자료에 없으면 **"모른다"고 답하기** (지어내기 금지)
3. 금액·기간·비율 등 숫자는 자료 그대로 정확하게
4. 항상 **출처(자료 제목) 표기**

여기에 더해, 세법·지원사업 정보의 특성(수시 개정)을 고려해 **모든 답변 끝에
'최신 기준 확인' 안내와 '세무·법률 자문 아님' 고지를 코드로 자동 부착**합니다.
LLM에게 맡기지 않고 코드가 강제하므로 누락될 수 없습니다.

> 💡 API 키가 없는 환경에서도 시연할 수 있도록, 키가 없으면 검색된 규정 원문을
> 정리해 보여주는 **검색 결과 안내 모드**로 자동 전환됩니다.

#### 멀티턴 대화 (대화 맥락 유지)

매 질문을 독립적으로 검색하면, *"미용실 창업 뭐부터 준비할까?"* 다음에 나온 *"자격증은 뭘 따야 해?"*
같은 **후속 질문에서 맥락(미용실)이 사라져** 엉뚱한 결과가 나옵니다. 실제로 "자격증은 뭘 따야 해?"를
단독 검색하면 미용실 면허 문서가 임계값 아래로 밀려 거절됩니다. 두 가지로 맥락을 잇습니다.

1. **검색 질의 재작성(query rewriting)** — `condense_query()`: 이전 대화를 LLM에 주고
   *"미용실 창업에 필요한 자격증은?"* 같은 **독립형 질문으로 변환**한 뒤 검색합니다.
   주제가 바뀐 새 질문은 LLM이 그대로 두므로, 단순히 이전 질문을 합칠 때 생기는 노이즈를 피합니다.
2. **대화 이력 전달** — `build_history_messages()`: 생성 단계에서 최근 대화를 LLM 메시지에 함께 넣어
   답변이 자연스럽게 이어지게 합니다. (출처·고지문은 떼어 토큰을 절약)

> API 키가 없으면 재작성 대신 **직전 질문을 합치는 간단 방식**으로 대체합니다(후속 질문에는 충분히 동작).

In [ ]:
SYSTEM_PROMPT = """당신은 창업하는 사장님을 돕는 법률·세무 안내 챗봇 '사장님, 전설이 되다'입니다.

[답변 규칙]
1. 반드시 아래 [참고 자료]의 내용만 근거로 답변합니다.
2. [참고 자료]에 없는 내용은 절대 추측하거나 지어내지 않습니다. 자료에 없으면
   "제가 가진 자료에서는 확인할 수 없는 내용이에요."라고 답합니다.
3. 금액·기간·비율·업종코드 등 숫자는 참고 자료에 적힌 그대로 정확하게 답합니다.
4. 한국어로 2~5문장 이내, 처음 창업하는 사장님도 이해할 수 있게 쉽고 친절하게 답합니다.
5. 답변 끝에 근거 문서를 (출처: 문서 제목) 형식으로 표기합니다. "자료 1" 같은 번호가 아니라
   실제 문서 제목(예: 출처: 등록 의무)을 적습니다."""

SYSTEM_PROMPT_NO_GUARDRAIL = "당신은 창업과 사업에 대해 답변하는 친절한 챗봇입니다. 질문에 자유롭게 답하세요."

INTRO_MESSAGE = ("안녕하세요 사장님! 저는 창업 법률·세무 안내 챗봇 **사장님, 전설이 되다**예요. 🏆\n"
                 "사업자등록, 부가가치세·종합소득세, 통신판매업 신고, 창업 세액감면, 정부 지원사업까지 —\n"
                 "국세청·중소벤처기업부 공식 자료를 근거로만 답해 드려요. 무엇이 궁금하세요?")
REFUSAL_MESSAGE = ("죄송해요 사장님, 제가 가진 공식 자료(국세청·중소벤처기업부)에서는 관련 내용을 찾지 못했어요. 🙏\n"
                   "국세상담센터(국번없이 126), 중소기업 통합콜센터(1357), 정부24에 문의하시면 정확한 안내를 받으실 수 있어요.")
DISCLAIMER = ("\n\n⚠️ 세법·지원사업 요건은 수시로 개정됩니다. 실제 신고·신청 전에 국세청 홈택스, 정부24, "
              "K-Startup에서 최신 기준을 꼭 확인하세요. 본 답변은 일반 정보 안내이며 세무·법률 자문이 아닙니다.")

# 자료를 찾지 못했을 때(임계값 미달) LLM이 '말투만' 다듬어 응대하기 위한 프롬프트.
# 잡담이면 공감, 정보 질문이면 거절+기관 안내 — 사실 정보 생성은 금지해 할루시네이션 방지 원칙 유지.
NO_CONTEXT_PROMPT = """당신은 창업 안내 챗봇 '사장님, 전설이 되다'입니다.
지금 사용자의 입력과 관련된 참고 자료를 찾지 못한 상태입니다. 다음 규칙으로 짧게(1~3문장) 답하세요.
1. 인사·감정 표현·잡담(예: 배고파, 힘들다, 떨려)이면 따뜻하게 공감해 주고, 당신이 도울 수 있는 주제
   (사업자등록·세금·통신판매업 신고·창업 세액감면·정부 지원사업)를 가볍게 알려 주세요.
2. 정보를 묻는 질문이면 가진 자료에 없어 정확히 답할 수 없다고 솔직히 말하고,
   국세상담센터(국번없이 126), 중소기업 통합콜센터(1357), 정부24를 안내하세요.
3. 어떤 경우에도 법령·세금·지원사업의 구체적 수치·절차를 기억에 의존해 답하지 마세요. 지어내면 안 됩니다.
4. 한국어로 친근하게, 사용자를 '사장님'이라고 부르세요."""

SMALLTALK_EXACT = {"하이", "ㅎㅇ", "헬로", "안녕"}    # 이 단어 '단독'일 때만 인사로 처리
SMALLTALK_PATTERNS = ["안녕하", "반가워", "반갑습니다", "고마워", "감사합니다", "감사해요",
                      "누구야", "누구니", "누구세요", "뭐 할 수", "뭘 할 수", "자기소개"]

def is_smalltalk(q):
    """짧은 인사/잡담 감지. '안녕'이 포함된 일반 질문을 오분류하지 않도록
    ①인사 단어 단독 ②짧은 문장(15자 이하)+인사 패턴 두 경우만 인정"""
    qs = q.strip().rstrip("!?.~^ ")
    return qs in SMALLTALK_EXACT or (len(qs) <= 15 and any(p in qs for p in SMALLTALK_PATTERNS))

# 감정 표현(배고파, 힘들어 등)은 정보 거절 문구 대신 공감으로 응대 — 사용자 테스트 피드백 반영
EMOTION_PATTERNS = ["배고파", "배고프", "힘들", "피곤", "졸려", "졸리", "심심", "지쳤", "지친다", "떨려", "긴장"]
EMOTION_MESSAGE = ("사장님, 창업 준비하시느라 고생이 많으세요! 잠깐 쉬어 가도 괜찮아요. ☕\n"
                   "기운 차리시면 사업자등록·세금·지원사업처럼 제가 도울 수 있는 것들, 언제든 물어봐 주세요!")

def is_emotion(q):
    qs = q.strip()
    return len(qs) <= 15 and any(p in qs for p in EMOTION_PATTERNS)

def call_llm(system_prompt, user_prompt, history_msgs=None, temperature=0.2, max_tokens=500):
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    messages = [{"role": "system", "content": system_prompt}]
    if history_msgs:                       # 멀티턴: 이전 대화를 system과 현재 질문 사이에 삽입
        messages += history_msgs
    messages.append({"role": "user", "content": user_prompt})
    resp = client.chat.completions.create(
        model=LLM_MODEL, messages=messages,
        temperature=temperature,   # 사실 기반 답변이므로 낮게 설정
        max_tokens=max_tokens,
    )
    return resp.choices[0].message.content.strip()

# ── 멀티턴(대화 맥락 유지) 지원 ──
def msg_text(content):
    """Gradio 버전마다 메시지 content가 str / list / dict로 달라짐(예: Gradio 6은
    [{'text':..., 'type':'text'}] 리스트). 어떤 형식이든 안전하게 순수 텍스트만 추출"""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return " ".join(msg_text(c) for c in content).strip()
    if isinstance(content, dict):
        return content.get("text") or content.get("value") or ""
    return str(content) if content is not None else ""

CONDENSE_SYSTEM = ("당신은 대화형 검색 도우미입니다. 아래 이전 대화를 참고하여, 사용자의 '현재 질문'을 "
                   "앞 맥락 없이도 그 자체로 이해되는 독립적인 검색 질문 한 문장으로 바꾸세요. "
                   "현재 질문이 이미 독립적이면 그대로 두세요. 다른 설명 없이 질문 문장만 출력하세요.")

def condense_query(question, history):
    """후속 질문을 '이전 맥락이 반영된 독립형 검색 질문'으로 변환 (멀티턴의 핵심).
    - LLM 모드: 대화를 보고 질의 재작성(query rewriting) — 후속/주제전환 모두 똑똑하게 처리
    - 키 없음: 직전 사용자 질문 1개를 붙여 맥락 보강 (간단 fallback)"""
    prev_users = [msg_text(m.get("content")) for m in history if m.get("role") == "user"]
    prev_users = [u for u in prev_users if u]
    if not prev_users:                          # 첫 질문이면 재작성 불필요
        return question
    if OPENAI_API_KEY:
        convo = "\n".join(f"- {u}" for u in prev_users[-3:]) + f"\n- (현재) {question}"
        try:
            return call_llm(CONDENSE_SYSTEM, f"[이전 대화]\n{convo}\n\n[독립형 검색 질문]",
                            temperature=0.0, max_tokens=80)
        except Exception:
            pass
    return prev_users[-1] + " " + question      # fallback: 직전 질문 + 현재 질문

def build_history_messages(history, max_turns=3):
    """생성 LLM에 넘길 최근 대화. assistant 답변에 붙은 출처(📚)/고지문(⚠️)은 떼어 토큰을 아낌"""
    msgs = []
    for m in history[-max_turns * 2:]:
        content = msg_text(m.get("content"))
        if m.get("role") == "assistant":
            content = content.split("📚")[0].split("⚠️")[0].strip()
        if content:
            msgs.append({"role": m["role"], "content": content})
    return msgs

def extractive_answer(hits):
    """API 키가 없을 때: 검색된 공식 자료 원문을 정리해 안내"""
    lines = ["사장님 질문과 관련된 공식 자료를 찾았어요. 📋\n"]
    for i, h in enumerate(hits, 1):
        lines.append(f"**{i}. {h['title']}** ({h['category']})\n{h['chunk']}\n")
    return "\n".join(lines)

def rag_answer(question, history=None, top_k=None, threshold=None, use_guardrail=True):
    """RAG 파이프라인 본체: (멀티턴)질의 재작성 → 검색 → 임계값 검사 → 프롬프트(+대화이력) → 생성 → 고지"""
    history = history or []
    top_k = TOP_K if top_k is None else top_k
    threshold = SIM_THRESHOLD if threshold is None else threshold

    question = question.strip()
    if not question:
        return "질문을 입력해 주세요.", []
    if is_smalltalk(question):                      # 인사·잡담은 검색 없이 응대
        return INTRO_MESSAGE, []
    if is_emotion(question):                        # 감정 표현엔 공감 먼저
        return EMOTION_MESSAGE, []

    search_query = condense_query(question, history)      # ★ 멀티턴: 이전 맥락을 반영해 검색 질의 보강
    hits = retriever.search(search_query, top_k=top_k)
    hits = [h for h in hits if h["score"] >= threshold]   # ★ 할루시네이션 방지 1차 관문
    if not hits:
        # 자료가 없으면 '사실 답변'은 하지 않는다. 키가 있으면 LLM이 잡담/정보질문을 구분해
        # 말투만 자연스럽게 응대 (NO_CONTEXT_PROMPT가 사실 정보 생성을 금지)
        if OPENAI_API_KEY:
            try:
                return call_llm(NO_CONTEXT_PROMPT, question,
                                history_msgs=build_history_messages(history)), []
            except Exception:
                return REFUSAL_MESSAGE, []
        return REFUSAL_MESSAGE, []

    if not OPENAI_API_KEY:                          # 키 없음 → 검색 결과 안내 모드
        return extractive_answer(hits) + DISCLAIMER, hits

    context = "\n\n".join(f"[자료 {i}] 제목: {h['title']} (분류: {h['category']})\n{h['chunk']}"
                            for i, h in enumerate(hits, 1))
    system = SYSTEM_PROMPT if use_guardrail else SYSTEM_PROMPT_NO_GUARDRAIL
    user = f"[참고 자료]\n{context}\n\n[질문]\n{question}"
    try:
        return call_llm(system, user, history_msgs=build_history_messages(history)) + DISCLAIMER, hits
    except Exception as e:
        return f"⚠️ LLM 호출 오류({e}). 검색 결과로 대신 안내해요.\n\n" + extractive_answer(hits) + DISCLAIMER, hits

# 동작 확인 ① 단일 질문
for q in ["안녕하세요!", "아 배고파", "창업하면 세금 감면 얼마나 받을 수 있어?", "로또 번호 추천해줘"]:
    ans, hits = rag_answer(q)
    print(f"Q: {q}\nA: {ans[:220]}{'...' if len(ans) > 220 else ''}\n{'-'*60}")

# 동작 확인 ② 멀티턴 — 이전 대화를 기억하고 후속 질문에 답하는지
print("\n[멀티턴 데모] '미용실' 맥락을 유지한 채 주어 없는 후속 질문 처리")
demo_hist = []
for q in ["미용실 창업하려고 하는데 뭐부터 준비할까?", "위생교육도 받아야 해?"]:
    ans, hits = rag_answer(q, history=demo_hist)
    demo_hist += [{"role": "user", "content": q}, {"role": "assistant", "content": ans}]
    print(f"Q: {q}\n   → 검색된 문서: {hits[0]['title'] if hits else '(거절)'}")

### 3.5 하이퍼파라미터 튜닝

RAG의 품질은 **검색이 정답 문서를 찾아오는가**에 좌우됩니다. 다음 3가지 하이퍼파라미터를
**정답이 표시된 평가셋**으로 정량 평가하여 결정합니다.

| 하이퍼파라미터 | 의미 | 트레이드오프 |
|---|---|---|
| `CHUNK_SIZE` | 청크 최대 길이 | 작으면 정보가 쪼개지고, 크면 임베딩이 잘림 |
| `TOP_K` | LLM에 넘길 검색 결과 수 | 작으면 정답을 놓치고, 크면 무관한 자료가 섞임 |
| `SIM_THRESHOLD` | 답변 거절 유사도 기준 | 낮으면 엉뚱한 질문에도 답하고(할루시네이션 위험), 높으면 정상 질문도 거절 |

**평가 지표 — 정답 키워드 기반 (answer recall)**
- **Hit@k**: 상위 k개 검색 청크 중 **정답 키워드(예: "20일", "525104")를 포함한 청크**가 있는 비율
- **MRR**(Mean Reciprocal Rank): 정답 키워드가 처음 등장하는 검색 순위의 역수 평균 (1등이면 1.0)

> 📐 **왜 '문서 번호 일치'가 아니라 '정답 키워드 포함'으로 평가하나?** v2 데이터에는 같은 내용을
> 다루는 문서가 여럿 존재합니다(예: 부가세 신고 기한은 세무안내·세탁소 가이드·음식점 가이드에 모두 등장).
> 어느 문서를 가져오든 **정답 정보를 가져왔다면 검색은 성공**이므로, 특정 문서 번호가 아닌
> 정답 키워드 포함 여부로 평가하는 것이 목적에 맞습니다.

> ⚠️ **튜닝/검증 분리**: 아래 18문항 평가셋은 하이퍼파라미터 **튜닝**에만 사용합니다.
> 튜닝에 한 번도 쓰지 않은 **별도 검증 질문(holdout)** 으로 5.5절에서 최종 성능을 다시 측정해
> "튜닝셋에만 과적합된 결과"가 아님을 확인합니다.

In [ ]:
# 평가셋: (질문, 정답에 반드시 포함되어야 할 키워드)
# 실제 사장님이 물어볼 법한 구어체 표현으로 작성 (문서 원문과 표현이 다르도록!)
# 키워드는 질문에는 없고 정답에만 있는 '핵심 사실 정보'로 선정
eval_set = [
    {"q": "사업 시작하면 사업자등록 언제까지 해야 해?",      "keyword": "20일"},
    {"q": "SNS로 물건 팔 건데 업종코드 뭐로 등록해?",        "keyword": "525104"},
    {"q": "사업자등록 안 하면 어떤 불이익이 있어?",          "keyword": "가산세"},
    {"q": "일반과세자랑 간이과세자는 뭐가 달라?",            "keyword": "1억 4백만원"},
    {"q": "간이과세자는 부가세 신고 언제 해?",               "keyword": "1월"},
    {"q": "매출이 얼마 미만이면 부가세 안 내도 돼?",         "keyword": "4,800만원"},
    {"q": "종합소득세 신고 기간 알려줘",                     "keyword": "5월"},
    {"q": "처음 사업하는데 장부는 어떻게 기록해야 해?",      "keyword": "간편장부"},
    {"q": "손님이 요청 안 해도 현금영수증을 끊어줘야 하는 경우가 있어?", "keyword": "10만원"},
    {"q": "통신판매업 신고는 어디서 어떻게 해?",             "keyword": "정부24"},
    {"q": "통신판매업 신고 면제되는 경우도 있어?",           "keyword": "50회"},
    {"q": "창업하면 세금 감면 몇 년 동안 받을 수 있어?",     "keyword": "5년"},
    {"q": "청년창업 세액감면 나이 조건이 어떻게 돼?",        "keyword": "34세"},
    {"q": "창업 세액감면에도 한도가 있어?",                  "keyword": "5억원"},
    {"q": "예비창업자한테 맞는 정부 지원사업 뭐가 있어?",    "keyword": "예비창업패키지"},
    # ▼ v2에서 보강된 업종가이드 영역 평가 질문
    {"q": "네일샵 차리려면 위생교육 몇 시간 받아야 해?",     "keyword": "3시간"},
    {"q": "동업으로 같이 창업할 때 뭘 조심해야 해?",         "keyword": "동업계약서"},
    {"q": "푸드트럭 영업은 아무 데서나 할 수 있어?",         "keyword": "허용"},
]

def evaluate_retriever(r, eval_set, k):
    """키워드 기반 Hit@k와 MRR: 상위 k개 청크 중 정답 키워드를 포함한 청크의 순위로 계산"""
    hit, rr = 0, 0.0
    for ex in eval_set:
        results = r.search(ex["q"], top_k=k)
        rank = next((i for i, h in enumerate(results, 1)
                     if ex["keyword"].lower() in h["chunk"].lower()), None)
        if rank:
            hit += 1
            rr += 1.0 / rank
    n = len(eval_set)
    return {"Hit@k": hit / n, "MRR": rr / n}

print(f"평가셋: {len(eval_set)}문항 준비 완료")

In [ ]:
# (a) CHUNK_SIZE 튜닝: 100자 / 180자 / 300자 비교
chunk_results = []
retrievers_by_cs = {}
for cs in [100, 180, 300]:
    r = Retriever(make_chunks(df, cs), embedder)
    retrievers_by_cs[cs] = r
    m = evaluate_retriever(r, eval_set, k=3)
    # 실제 임베딩되는 텍스트("제목: 청크")의 서브워드 토큰 길이 측정 → 임베딩 잘림 여부 확인
    emb_texts = (r.chunk_df["title"] + ": " + r.chunk_df["chunk"]).tolist()
    tok_lens = [len(hf_tokenizer.encode(t)) for t in emb_texts]
    chunk_results.append({"CHUNK_SIZE": f"{cs}자", "청크 수": r.index.ntotal,
                          "최대 토큰": max(tok_lens),
                          "128토큰 초과 청크": int(sum(l > MAX_SEQ_LEN for l in tok_lens)),
                          "Hit@3": m["Hit@k"], "MRR": m["MRR"]})

chunk_result_df = pd.DataFrame(chunk_results)
display(chunk_result_df.round(3))

ax = chunk_result_df.plot(x="CHUNK_SIZE", y=["Hit@3", "MRR"], kind="bar", figsize=(8, 4), rot=0,
                          color=["#4C72B0", "#DD8452"])
ax.set_title("청크 크기별 검색 성능")
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

In [ ]:
# (b) TOP_K 튜닝: 검색 결과를 몇 개까지 LLM에 넘길 것인가
#
# (a) 결과 해석:
#   - 100자: 잘림은 없지만 문맥이 잘게 쪼개져 검색 성능(Hit/MRR)이 가장 낮음
#   - 300자: Hit는 근소하게 높지만 청크의 70% 이상이 128토큰을 넘어 임베딩이 잘림 (2.4절 문제 재현)
#     ※ 키워드 평가는 청크 '전문' 기준이라, 임베딩에서는 잘려나간 뒷부분의 키워드도
#       적중으로 집계되는 착시가 포함될 수 있음
#   - 180자: 잘림 1% 수준 + 문맥 유지의 균형점 → 선택
BEST_CHUNK_SIZE = 180
best_retriever = retrievers_by_cs[BEST_CHUNK_SIZE]

topk_results = []
for k in [1, 2, 3, 5, 7]:
    m = evaluate_retriever(best_retriever, eval_set, k=k)
    topk_results.append({"TOP_K": k, "Hit@k": m["Hit@k"], "MRR": m["MRR"]})

topk_result_df = pd.DataFrame(topk_results)
display(topk_result_df.round(3))

ax = topk_result_df.plot(x="TOP_K", y=["Hit@k", "MRR"], kind="line", marker="o", figsize=(8, 4),
                         color=["#4C72B0", "#DD8452"])
ax.set_title("TOP_K별 검색 성능: k가 커질수록 Hit는 오르지만 무관한 자료 유입도 증가")
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

In [ ]:
# (c) SIM_THRESHOLD 튜닝: '아는 질문'과 '모르는 질문'의 유사도 분포 분리점 찾기
ood_questions = [   # 도메인 외(Out-of-Domain) 질문: 챗봇이 거절해야 정상
    # 일상 질문 — 창업·세무와 무관한 주제
    "오늘 점심 뭐 먹을까?", "주말에 볼만한 영화 추천해줘", "손흥민 어제 경기 어땠어?",
    "김치찌개 맛있게 끓이는 법", "아이폰 신형 언제 나와?", "요즘 유행하는 노래 뭐야?",
    "오늘 날씨 어때?", "넷플릭스에서 뭐 볼까?",
]
# 참고: v1에서 도메인 외였던 '가게 자리', '권리금', '인스타 마케팅' 등 사업 인접 질문은
# v2 업종가이드 보강(입지 선정·임대차계약·인터넷쇼핑몰 창업)으로 '답할 수 있는 질문'이 되어
# 도메인 외 목록에서 제외했습니다 → 5.2절 '⑤ 데이터 보강으로 답변 가능해진 질문'에서 확인

in_scores  = [best_retriever.search(ex["q"], top_k=1)[0]["score"] for ex in eval_set]
ood_scores = [best_retriever.search(q, top_k=1)[0]["score"] for q in ood_questions]

# 임계값을 사람이 정하지 않고, 두 분포의 중간 지점으로 데이터가 결정하게 함
suggested_threshold = round((min(in_scores) + max(ood_scores)) / 2, 2)

plt.figure(figsize=(9, 4.5))
plt.hist(in_scores,  bins=12, alpha=0.65, label="도메인 내 질문 (답해야 함)", color="#4C72B0")
plt.hist(ood_scores, bins=12, alpha=0.65, label="도메인 외 질문 (거절해야 함)", color="#C44E52")
plt.axvline(suggested_threshold, color="black", ls="--", lw=2,
            label=f"분석으로 결정된 임계값 = {suggested_threshold}")
plt.xlabel("최고 유사도 점수 (top-1)")
plt.ylabel("질문 수")
plt.title("질문 유형별 최고 유사도 분포: 임계값으로 두 분포를 분리")
plt.legend()
plt.tight_layout()
plt.show()

print(f"도메인 내 질문 top-1 유사도: 최소 {min(in_scores):.3f} / 평균 {np.mean(in_scores):.3f}")
print(f"도메인 외 질문 top-1 유사도: 최대 {max(ood_scores):.3f} / 평균 {np.mean(ood_scores):.3f}")
print(f"→ 두 분포의 중간 지점 {suggested_threshold}를 임계값으로 채택 (다음 셀에서 확정)")

In [ ]:
# 튜닝 결과를 반영해 최종 하이퍼파라미터 확정 & 리트리버 재구축
CHUNK_SIZE    = BEST_CHUNK_SIZE
# (b) 해석: v1(문서 30건)에서는 TOP_K=3으로 충분했지만, v2(726건)는 같은 주제를 다루는
# 업종별 문서가 많아 정답 청크가 4~5위로 밀리는 경우가 생김 → Hit@k가 k=5에서 수렴.
# "데이터가 바뀌면 하이퍼파라미터도 다시 튜닝해야 한다"는 것을 보여주는 사례.
# (무관한 자료가 함께 검색되더라도 유사도 임계값이 1차로 걸러줌)
TOP_K         = 5
SIM_THRESHOLD = suggested_threshold   # (c) 분포 분석이 결정한 값
assert max(ood_scores) < SIM_THRESHOLD < min(in_scores), "임계값은 두 분포 사이에 있어야 함"

retriever = best_retriever            # (a)에서 구축한 180자 리트리버 재사용 (재임베딩 생략)
final_metrics = evaluate_retriever(retriever, eval_set, k=TOP_K)

print("=" * 50)
print(f" 최종 하이퍼파라미터")
print(f"   CHUNK_SIZE    = {CHUNK_SIZE}자")
print(f"   TOP_K         = {TOP_K}")
print(f"   SIM_THRESHOLD = {SIM_THRESHOLD}")
print(f" 최종 검색 성능: Hit@{TOP_K} = {final_metrics['Hit@k']:.1%}, MRR = {final_metrics['MRR']:.3f}")
print("=" * 50)

---
## 4. Gradio UI 챗봇 구현

완성된 RAG 파이프라인을 **Gradio** 채팅 UI로 감쌉니다.

- 답변 아래에 **참고한 공식 자료(출처·원자료 기관)와 유사도 점수**를 함께 표시 → 신뢰성 확보
- **멀티턴 대화**: Gradio가 누적한 대화 기록(`chat_history`)을 그대로 `rag_answer`에 넘겨,
  "미용실 창업 뭐부터?" → "자격증은?" 같은 후속 질문도 맥락을 이어 답합니다 (3.4절)
- 모든 답변에 **최신 기준 확인 + 자문 아님 고지**가 자동 부착 (3.4절 `DISCLAIMER`)
- **검색 설정 패널**에서 `TOP_K`, `SIM_THRESHOLD`를 실시간 조절 가능 → 발표 때 하이퍼파라미터의 효과를 라이브로 시연할 수 있습니다 (예: 임계값을 0으로 내리면 엉뚱한 질문에도 답하는 모습)
- **UI 디자인**: '전설' 컨셉의 네이비×골드 테마 — Gradio 테마(`gr.themes.Soft`) 커스터마이징 +
  CSS로 히어로 배너·커버리지 칩·말풍선을 직접 스타일링. 컴포넌트 인자는 Gradio 버전(4/5/6)에 따라
  다를 수 있어 **미지원 인자를 자동으로 제거하는 호환 로직**을 넣었습니다.
- **라이트/다크 모드 모두 가시성 보장**: 말풍선 글자색은 강제하지 않고 테마에 맡기며, 배경은
  반투명 틴트(rgba)와 **Gradio 테마 변수**(`var(--background-fill-secondary)` 등)만 사용 —
  사용자의 화면 모드가 무엇이든 대비가 유지됩니다.

In [ ]:
import gradio as gr

def format_sources(hits):
    """중복 문서를 제거하고 출처 목록 생성 (원자료 기관 포함)"""
    seen, lines = set(), []
    for h in hits:
        if h["doc_id"] in seen:
            continue
        seen.add(h["doc_id"])
        lines.append(f"- [{h['category']}] **{h['title']}** — {h['source']} (유사도 {h['score']:.2f})")
    return "\n".join(lines)

def respond(message, chat_history, top_k, threshold):
    if not message.strip():               # 빈 입력은 무시 (빈 말풍선 방지)
        return "", chat_history
    # chat_history는 직전까지의 대화(현재 메시지 추가 전) → 그대로 멀티턴 맥락으로 전달
    answer, hits = rag_answer(message, history=chat_history, top_k=int(top_k), threshold=float(threshold))
    if hits:
        answer += "\n\n---\n📚 **참고한 공식 자료**\n" + format_sources(hits)
    chat_history = chat_history + [
        {"role": "user", "content": message},
        {"role": "assistant", "content": answer},
    ]
    return "", chat_history

# ─── '전설' 컨셉 테마: 네이비 × 골드 ───
THEME = gr.themes.Soft(
    primary_hue="amber",
    neutral_hue="slate",
    font=[gr.themes.GoogleFont("Noto Sans KR"), "ui-sans-serif", "system-ui", "sans-serif"],
)

CSS = """
.gradio-container {max-width: 960px !important; margin: 0 auto !important;}
/* 히어로 배너 — 배경과 글자색을 한 쌍으로 직접 지정한 독립 영역이라 어떤 테마에서도 동일하게 보임 */
#hero {background: linear-gradient(135deg, #1c2844 0%, #2b3d68 60%, #8a6d1f 135%);
       border-radius: 18px; padding: 26px 30px 22px; color: #f4f6fb; margin-bottom: 6px;}
#hero h1 {color: #ffd24d; margin: 0 0 8px; font-size: 1.85em; letter-spacing: -0.5px;}
#hero p  {color: #e9edf7; margin: 0; line-height: 1.6; font-size: 0.95em;}
#hero b  {color: #ffe9a8;}
#hero .chip {display: inline-block; background: rgba(255,255,255,.12);
             border: 1px solid rgba(255,210,77,.45); border-radius: 999px;
             padding: 3px 13px; margin: 10px 6px 0 0; font-size: .8em; color: #ffe9a8;}
/* 채팅 카드 */
#chatbox {border-radius: 16px; box-shadow: 0 6px 22px rgba(0,0,0,.12);}
/* 말풍선 — 글자색은 건드리지 않고(테마가 모드별 대비를 보장) 배경만 입힘:
   사용자 = 반투명 골드 틴트(라이트/다크 모두 자연스러움), 챗봇 = 테마 변수 사용 */
.message.user, .user-row .message {background: rgba(240,178,41,.16) !important;
                                   border: 1px solid rgba(240,178,41,.38) !important;}
.message.bot, .bot-row .message {background: var(--background-fill-secondary) !important;
                                 border: 1px solid var(--border-color-primary) !important;}
/* Gradio 6은 말풍선 안에 .message가 한 번 더 중첩되어 박스가 두 겹으로 보임
   → 안쪽 것은 투명 처리 (#chatbox ID로 특이도를 높여 리렌더링 후에도 항상 적용) */
#chatbox .message .message {background: transparent !important; border: none !important;
                            box-shadow: none !important;}
/* 입력줄·버튼 — 골드 배경 + 진갈색 글자를 한 쌍으로 고정 (두 모드 모두 대비 충분) */
#send-btn {background: linear-gradient(135deg, #f3b62b, #dc9117) !important;
           color: #2b2105 !important; font-weight: 700; border: none !important;
           border-radius: 12px !important;}
#notice {color: var(--body-text-color-subdued); font-size: .8em; text-align: center; margin-top: 2px;}
footer {visibility: hidden;}  /* 기본 Gradio 푸터 숨김 */
"""

CHAT_PLACEHOLDER = (
    "<div style='text-align:center; color: var(--body-text-color-subdued);'>"
    "<div style='font-size:2.6em'>🏆</div>"
    "<b style='font-size:1.15em; color: var(--body-text-color);'>무엇이든 물어보세요, 사장님!</b><br>"
    "사업자등록 · 세금 · 인허가 · 세액감면 · 지원사업 · 업종별 창업 절차<br>"
    "<span style='font-size:.85em'>모든 답변에 공식 자료 출처가 함께 표시됩니다</span></div>"
)

def make_component(cls, **kwargs):
    """Gradio 버전(4/5/6)에 따라 없는 인자는 자동으로 빼고 컴포넌트 생성"""
    import re as _re
    while True:
        try:
            return cls(**kwargs)
        except TypeError as e:
            m = _re.search(r"'(\w+)'", str(e))
            if m and m.group(1) in kwargs:
                kwargs.pop(m.group(1))      # 미지원 인자 제거 후 재시도
            else:
                raise

with gr.Blocks(title="사장님, 전설이 되다", theme=THEME, css=CSS) as demo:
    gr.HTML(
        '<div id="hero">'
        '<h1>🏆 사장님, 전설이 되다</h1>'
        '<p>창업이 처음인 사장님을 위한 법률·세무 안내 챗봇 — 국세청·중소벤처기업부·법제처 '
        '<b>공식 자료 726건</b>을 검색해 근거와 출처를 들어 답합니다.</p>'
        '<div>'
        '<span class="chip">사업자등록</span><span class="chip">부가세·종소세</span>'
        '<span class="chip">통신판매업 신고</span><span class="chip">창업 세액감면</span>'
        '<span class="chip">정부 지원사업</span><span class="chip">업종별 가이드 23종</span>'
        '</div></div>'
    )
    chatbot = make_component(
        gr.Chatbot, type="messages", height=440, label="상담 내용",
        elem_id="chatbox", show_copy_button=True, placeholder=CHAT_PLACEHOLDER,
    )
    with gr.Row():
        msg = gr.Textbox(placeholder="예: 청년 창업하면 세금 감면 얼마나 받아요?",
                         scale=5, container=False, autofocus=True)
        send_btn = gr.Button("질문하기 📨", variant="primary", scale=1, elem_id="send-btn")
    gr.Examples(
        examples=["사업자등록은 언제까지 해야 해?", "일반과세자랑 간이과세자는 뭐가 달라?",
                  "청년창업 세액감면 나이 조건이 어떻게 돼?", "네일샵 차리려면 위생교육 받아야 해?",
                  "직원 뽑으면 4대보험 어떻게 해?", "오늘 점심 뭐 먹을까?"],
        inputs=msg, label="💬 예시 질문 (4~5번째: v2 업종가이드로 보강된 질문 / 마지막: 거절 동작 확인용)",
    )
    with gr.Accordion("🔧 검색 설정 (하이퍼파라미터 라이브 데모)", open=False):
        topk_slider = gr.Slider(1, 7, value=TOP_K, step=1, label="TOP_K — 검색할 자료 수")
        th_slider = gr.Slider(0.0, 0.8, value=SIM_THRESHOLD, step=0.01,
                              label="SIM_THRESHOLD — 답변 거절 임계값 (0으로 내리면 무조건 답변 시도)")
    gr.HTML('<p id="notice">⚠️ 본 챗봇은 일반 정보 안내 서비스이며 세무·법률 자문이 아닙니다. '
            '신고·신청 전 국세청(126)·정부24·K-Startup에서 최신 기준을 확인하세요.</p>')

    msg.submit(respond, [msg, chatbot, topk_slider, th_slider], [msg, chatbot])
    send_btn.click(respond, [msg, chatbot, topk_slider, th_slider], [msg, chatbot])

# share=True로 바꾸면 외부 접속용 임시 링크 생성 (발표 시 유용)
try:        # Gradio 6+: theme/css는 launch()로 전달해야 적용됨
    demo.launch(share=False, theme=THEME, css=CSS)
except TypeError:   # Gradio 4~5: Blocks 생성자에서 이미 적용되므로 그대로 실행
    demo.launch(share=False)

---
## 5. 테스트 및 챗봇 성능 개선

### 5.1 할루시네이션 방지 전략 (3중 안전장치 + 고지문)

| 단계 | 전략 | 구현 위치 |
|---|---|---|
| ① 검색 단계 | **유사도 임계값** 미달 시 사실 답변 차단 — 잡담이면 공감, 정보 질문이면 거절 + 전문기관(126·1357·정부24) 안내 | `rag_answer()`의 threshold 필터 + `NO_CONTEXT_PROMPT` |
| ② 생성 단계 | **가드레일 프롬프트**: "참고 자료에 있는 내용만, 없으면 모른다고" | `SYSTEM_PROMPT` 규칙 1·2 |
| ③ 표시 단계 | 모든 답변에 **출처(원자료 기관)·유사도 표시** → 사용자가 직접 검증 가능 | Gradio `format_sources()` |
| ④ 고지 단계 | **최신 기준 확인 + 세무·법률 자문 아님** 고지를 코드가 모든 답변에 자동 부착 | `DISCLAIMER` 상수 |

> 핵심 설계 철학: **"모른다고 답하는 것이 틀린 답을 하는 것보다 낫다"** — 세무·법률 정보는
> 틀리면 가산세·과태료 등 실제 금전 피해로 이어지기 때문입니다.

> 🔁 **테스트 → 개선의 선순환 (개선 이력)**
> 1. **데이터 보강 (v1→v2)**: v1 테스트에서 '직원 고용(4대보험)', '가게 입지', '임대차' 질문이
>    거절되는 공백을 발견 → 법제처 업종가이드 23종을 크롤링해 보강. 거절 로그가 수집 목록이 됨 (5.2 ⑤에서 검증)
> 2. **말투 개선**: "아 배고파" 같은 감정 표현에 고정 거절 문구로 답해 차갑다는 사용자 피드백
>    → 감정 표현은 규칙 기반 공감(`EMOTION_MESSAGE`), 그 외 자료 없는 입력은 LLM이 잡담(공감)과
>    정보 질문(거절+기관 안내)을 구분해 응대(`NO_CONTEXT_PROMPT`). **수치·절차 생성은 금지**해
>    할루시네이션 방지 원칙 유지
> 3. **출처 표기 수정**: LLM이 "(출처: 자료 1)"처럼 내부 번호를 인용하던 문제 → 실제 문서 제목을
>    인용하도록 프롬프트 수정
> 4. **평가 지표 개선**: v2에서 같은 내용을 다루는 문서가 많아져 '문서 번호 일치' 평가가 부적절해짐
>    → '정답 키워드 포함 청크 검색' 기준으로 변경, TOP_K도 3→5로 재튜닝 (3.5절)

### 5.2 시나리오별 테스트 케이스

챗봇의 목적(1.5절 성공 기준)에 맞춰 5가지 유형의 질문으로 동작을 검증합니다.

- **⑤ 유형**은 v1 테스트에서 거절됐던 질문들 — **데이터 보강(v2)으로 답변 가능해졌는지** 확인합니다.
- **⑥ 유형(한계 케이스)** 은 정확한 자료가 없는데도 비슷한 문서가 임계값을 넘는 경우입니다
  (예: "법인 설립" 질문에 경비업 법인 허가 문서가 검색됨). 이때는 임계값(1차 방어)이 아니라
  **가드레일 프롬프트(2차 방어)가 "자료에 없다"고 답하도록 막는 역할**을 합니다 — LLM 생성 모드에서 동작을 확인하세요.

In [ ]:
test_cases = [
    ("① 데이터 내 질문",        "사업자등록은 언제까지 해야 하나요?"),
    ("① 데이터 내 질문",        "창업하면 세금 감면 얼마나 받을 수 있어?"),
    ("② 구어체/유의어",         "나라에서 창업자한테 지원해주는 돈 같은 거 있어?"),
    ("② 구어체/유의어",         "인스타에서 옷 팔려고 하는데 사업자등록 해야 돼?"),
    ("③ 데이터 외(일상)",       "환율이 왜 이렇게 올라?"),
    ("④ 인사/잡담",             "안녕! 너는 누구야?"),
    ("④ 인사/잡담",             "아 배고파"),
    ("⑤ v2 보강으로 답변 가능", "직원 뽑으면 4대보험 어떻게 가입해?"),
    ("⑤ v2 보강으로 답변 가능", "가게 자리는 어디에 잡는 게 좋을까?"),
    ("⑥ 한계 케이스",           "법인 설립 절차 알려줘"),
]

rows = []
for case_type, q in test_cases:
    ans, hits = rag_answer(q)
    rows.append({
        "유형": case_type,
        "질문": q,
        "최고 유사도": f"{hits[0]['score']:.3f}" if hits else "-",
        "검색 문서": hits[0]["title"] if hits else "(검색 결과 없음/생략)",
        "답변(앞부분)": ans[:80].replace("\n", " ") + ("..." if len(ans) > 80 else ""),
    })

pd.set_option("display.max_colwidth", 90)
test_result_df = pd.DataFrame(rows)
display(test_result_df)

print("\n✅ 체크 포인트")
print(" ① 데이터 내 질문   → 정답 문서를 찾아 정확히 답변 (+ 출처·고지문 자동 부착)")
print(" ② 구어체/유의어    → 단어가 달라도 의미 검색으로 정답 문서 매칭")
print(" ③ 데이터 외(일상)  → 임계값 미달로 정중히 거절 (할루시네이션 방지 1차 방어)")
print(" ④ 인사/잡담        → 검색 없이 자기소개/공감으로 자연스럽게 응대")
print(" ⑤ v1에서 거절됐던 질문 → 업종가이드 보강(v2)으로 이제 근거 자료와 함께 답변!")
print(" ⑥ 한계 케이스      → '법인 설립'은 정확한 자료가 없는데 유사 문서(경비업 법인 허가 등)가")
print("                      임계값을 넘음 → 가드레일 프롬프트(2차 방어)가 '자료에 없다'고 답해야 함")
print("\n📌 남은 데이터 보강 후보: 법인 설립 절차, 상표·지식재산, 프랜차이즈 가맹 계약")

### 5.3 성능 개선 전/후 비교: 안전장치가 없다면?

**창업과 무관한 일상 질문**("요즘 핫한 주식 종목")에 대해 ▲안전장치를 모두 끈 버전과
▲최종 버전의 응답을 비교합니다. 안전장치가 없으면 무관한 문서(지역상권·반찬가게 자료 등)라도
검색되는 대로 답변 근거로 사용하므로 **그럴듯한 오답(할루시네이션)** 위험이 생깁니다.

> ⚠️ API 키 없이 실행하면 두 버전 모두 '검색 원문 안내' 방식이므로 **① 임계값의 효과만** 시연됩니다.
> ② 가드레일 프롬프트의 효과(지어내기 억제)는 API 키를 설정해 LLM 생성 모드로 실행할 때 확인할 수 있습니다.

In [ ]:
danger_q = "요즘 핫한 주식 종목 알려줘"

mode = "LLM 생성 모드" if OPENAI_API_KEY else "검색 결과 안내 모드 (API 키 없음)"
print(f"질문: {danger_q}   [현재 실행 모드: {mode}]\n")
print("─" * 70)
print("【개선 전】 임계값 없음(threshold=0) + 가드레일 프롬프트 없음")
print("─" * 70)
ans_before, hits_before = rag_answer(danger_q, threshold=0.0, use_guardrail=False)
print(f"검색된 문서: {[h['title'][:30] for h in hits_before]}")
print(f"→ 질문(주식)과 무관한 문서가 그대로 답변 근거로 사용됨 (오답 위험)\n")
print(ans_before[:300])
print()
print("─" * 70)
print(f"【개선 후】 임계값 {SIM_THRESHOLD} + 가드레일 프롬프트 적용 (최종 버전)")
print("─" * 70)
ans_after, hits_after = rag_answer(danger_q)
print(ans_after)

### 5.4 답변 정확도 정량 평가 (정답 키워드 포함률)

평가셋 15문항 각각에 대해, **정답에 반드시 들어가야 할 핵심 정보(키워드)** 가
챗봇 답변에 포함되는지 자동 채점합니다. (예: "세금 감면 몇 년?" → 답변에 "5년" 포함 여부)

> 측정의 의미: API 키 없는 모드에서는 "검색이 정답 문서를 가져왔는가"(**검색 정확도**)를,
> LLM 모드에서는 "생성된 답변에 정답 정보가 들어 있는가"(**답변 정확도**)를 측정하게 됩니다.

In [ ]:
rows = []
for ex in eval_set:
    ans, hits = rag_answer(ex["q"])
    if ans.startswith("⚠️"):                       # LLM 호출 오류는 정답으로 집계하지 않음
        status = "⚠️ 오류"
    else:
        status = "✅" if ex["keyword"].lower() in ans.lower() else "❌"
    rows.append({"질문": ex["q"], "정답 키워드": ex["keyword"], "포함 여부": status})

acc_df = pd.DataFrame(rows)
accuracy = (acc_df["포함 여부"] == "✅").mean()
display(acc_df)
print(f"\n📊 정답 키워드 포함률: {accuracy:.1%} ({(acc_df['포함 여부'] == '✅').sum()}/{len(acc_df)}문항)")

### 5.5 검증셋(holdout) 평가 — 튜닝에 쓰지 않은 질문으로 일반화 확인

3.5절 튜닝에 사용한 질문으로만 성능을 보고하면 수치가 낙관적으로 부풀 수 있습니다(과적합 위험).
튜닝 과정에서 **한 번도 사용하지 않은 새 질문 6개**(도메인 내 4 + 도메인 외 2)로
최종 확정된 하이퍼파라미터의 일반화 성능을 검증합니다.

In [ ]:
holdout_set = [
    {"q": "현금영수증 발급 거부하면 어떻게 돼?",                 "keyword": "5%",   "expect": "answer"},
    {"q": "수도권과밀억제권역에서 청년이 창업하면 감면율 몇 프로야?", "keyword": "50%",  "expect": "answer"},
    {"q": "창업하면 인지세도 면제돼?",                           "keyword": "2년",  "expect": "answer"},
    {"q": "청년창업사관학교는 몇 살까지 지원 가능해?",            "keyword": "39세", "expect": "answer"},
    {"q": "서울에서 부산까지 KTX 요금 얼마야?",                  "keyword": None,   "expect": "refuse"},
    {"q": "비트코인 지금 사도 돼?",                              "keyword": None,   "expect": "refuse"},
]

rows = []
for ex in holdout_set:
    ans, hits = rag_answer(ex["q"])
    # 거절 판정은 답변 문구가 아니라 '근거 자료를 사용했는가'로 — LLM이 거절을 자연어로
    # 표현해도(말투 개선 반영) 자료 없이 사실 답변을 하지 않았다면 올바른 동작
    refused = len(hits) == 0
    if ex["expect"] == "answer":
        ok = (not refused) and ex["keyword"].lower() in ans.lower()
        expected = f"답변 (키워드: {ex['keyword']})"
    else:
        ok = refused
        expected = "거절/안내 (자료 미사용)"
    rows.append({"질문": ex["q"], "기대 동작": expected,
                 "실제": "거절/안내" if refused else "답변", "판정": "✅" if ok else "❌"})

holdout_df = pd.DataFrame(rows)
display(holdout_df)
n_ok = (holdout_df["판정"] == "✅").sum()
print(f"\n📊 검증셋(holdout) 정확도: {n_ok/len(holdout_df):.1%} ({n_ok}/{len(holdout_df)}문항)")

### 5.6 멀티턴 대화 테스트 — 후속 질문이 맥락을 잇는가

실제 상담은 한 번에 끝나지 않습니다. *"미용실 창업 뭐부터?"* → *"자격증은?"* 처럼 이어지죠.
이전 맥락을 무시하고 **현재 질문만 단독으로 검색했을 때**와, **대화 맥락을 반영했을 때**(3.4절 멀티턴)를
같은 후속 질문으로 비교합니다.

In [ ]:
# 후속 질문은 그 자체로는 주어(미용실)가 없어 맥락이 필요함
followups = ["위생교육도 받아야 해?", "시설 기준은 어떻게 돼?"]
prior_turn = {"role": "user", "content": "미용실 창업하려고 하는데 뭐부터 준비할까?"}

rows = []
for q in followups:
    # (A) 맥락 없이 현재 질문만 검색 (멀티턴 미적용)
    _, hits_solo = rag_answer(q, history=[])
    # (B) 직전 대화를 함께 전달 (멀티턴 적용)
    _, hits_ctx = rag_answer(q, history=[prior_turn])
    rows.append({
        "후속 질문": q,
        "맥락 없이 (단독 검색)": hits_solo[0]["title"] if hits_solo else "❌ 거절/엉뚱",
        "맥락 반영 (멀티턴)": hits_ctx[0]["title"] if hits_ctx else "❌ 거절",
    })

pd.set_option("display.max_colwidth", 40)
display(pd.DataFrame(rows))
print("→ 단독 검색은 '미용실'이라는 주어가 없어 거절되거나 다른 업종 문서로 새지만,")
print("  멀티턴은 이전 대화의 '미용실' 맥락을 반영해 미용실 관련 문서를 정확히 찾아냅니다.")

---
## 6. 결론 및 한계

### 구현 요약

| 평가 항목 | 구현 내용 |
|---|---|
| 문제 정의 | 흩어진 창업 법률·세무 정보 + 범용 LLM 할루시네이션 → 공식 자료 근거 창업 안내 챗봇 |
| 데이터·텍스트 분석 | 국세청·중기부·법제처 공식 자료 31개 카테고리 726개 문서(청크 3,089개에서 복원) / kiwipiepy 형태소 토큰화 / 빈도·길이 분석으로 청킹 근거 도출 |
| RAG 구성 | 문장 단위 청킹 → ko-sroberta 임베딩 → FAISS 리트리버 → 가드레일 프롬프트 → gpt-4o-mini |
| 하이퍼파라미터 튜닝 | CHUNK_SIZE·TOP_K·SIM_THRESHOLD를 키워드 기반 Hit@k/MRR + 유사도 분포로 정량 결정 (v2에서 TOP_K 3→5 재튜닝) |
| 성능 개선 | v1 테스트 공백 → v2 데이터 보강의 선순환 실증, 3중 할루시네이션 방지 + 고지문, 멀티턴 대화(질의 재작성 + 이력 전달), 시나리오 테스트, 키워드 정확도 + 별도 검증셋(holdout) 평가 |

### 한계 및 향후 개선 방향

1. **남은 데이터 공백** — v1 테스트에서 발견한 공백(4대보험·입지·임대차)은 v2 업종가이드로 보강했지만,
   **법인 설립 절차, 상표·지식재산, 프랜차이즈 가맹 계약**은 여전히 정확한 자료가 없음 → 다음 보강 후보.
   **거절된 질문 로그 → 데이터 수집 목록**의 선순환을 계속 돌리는 것이 이 설계의 운영 방식
2. **유사도 임계값의 한계 (실측)** — 일상 질문(top-1 최대 ~0.46)과 도메인 내 질문(최소 ~0.61)은 분리되지만,
   **"법인 설립"(0.65)처럼 자료가 없는 사업 질문은 유사 문서가 임계값을 넘어** 1차 방어가 뚫림 →
   가드레일 프롬프트(2차 방어)에 의존. 키워드 검색(BM25) 결합 **하이브리드 검색**, 재정렬(re-ranking)로 보완 가능
3. **크롤링 데이터 정제** — 법제처 자료의 표·서식이 텍스트로 펴지면서 일부 섹션 제목이 불완전함
   (예: "세탁소 - %", "제7호 ①번)."). 제목 정제와 표 구조 보존이 검색 품질을 더 올릴 여지
4. **멀티턴의 남은 한계** — 검색 질의 재작성 + 대화 이력 전달로 후속 질문은 처리하지만(5.6절),
   질의 재작성에 LLM 호출이 한 번 더 들어가 지연·비용이 늘고, 키 없는 모드의 fallback(직전 질문 합치기)은
   주제가 급변할 때 노이즈가 생길 수 있음. 대화가 길어질 때의 이력 요약(summary memory)도 향후 과제
5. **법령 개정 추적** — 고지문으로 1차 대응하지만, 원천 자료가 바뀌면 조원이 jsonl을 재크롤링해야 함.
   국세청·K-Startup 공고 모니터링 자동화가 향후 과제 (RAGAS 등 RAG 전용 평가 자동화도 병행 가능)